# GPU batch edge detection with CUDA NPP - Colab runner

This notebook builds and runs the project end to end on a free Colab GPU and
packages the evidence for the assignment's proof-of-execution upload.

**Before running:** *Runtime -> Change runtime type -> Hardware accelerator: GPU*
(a T4 is plenty; the whole notebook takes roughly five minutes).

Run the cells in order. The last cell downloads
`proof_of_execution.tar.gz` to your machine.

## 1. Confirm the runtime really has a GPU

If this cell prints an error, the runtime is CPU-only and nothing below will
produce valid evidence.

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Fetch the project

The repository carries its own 103-image USC-SIPI dataset, so there is
nothing else to download.

In [ ]:
%cd /content
!rm -rf CUDA-NPP-Edge-Detection
!git clone --depth 1 https://github.com/JackyJiang08/CUDA-NPP-Edge-Detection.git
%cd /content/CUDA-NPP-Edge-Detection
!ls data/input | wc -l

## 3. Install the two optional host-side tools

`cpplint` backs the Google C++ Style check and Pillow renders the contact
sheets. Neither is needed to build or run the pipeline itself.

In [ ]:
!pip install --quiet cpplint pillow

## 4. Build for this GPU

The Makefile defaults to a fat binary covering sm_70 through sm_86. Here we
read the compute capability off the attached device and build only that
target, which cuts the compile to a single pass.

In [ ]:
import subprocess

capability = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().strip().splitlines()[0]
arch = 'sm_' + capability.replace('.', '')
print('building for', arch)

!make clean
!make -j$(nproc) GENCODE="-arch={arch}"
!ls -la bin/

## 5. Sanity checks before the real run

Host tests, the style check, and the usage message.

In [ ]:
!make test
!make lint && echo 'cpplint: no findings'
!./bin/edge_pipeline --help

## 6. The full evidence run

`run.sh` does everything the assignment asks for: the whole 103-image set at
the default settings, a stage-dump run, a fixed-threshold run, a
GPU-versus-host-reference verification pass over every image, a stream
scaling sweep, error-handling checks, contact sheets, and the packaged
archive. Expect a few minutes, most of it in the host reference engine.

The build step inside `run.sh` re-runs `make`, which is already up to date,
so it costs nothing.

In [ ]:
!GENCODE="-arch={arch}" ./run.sh 2>&1 | tail -60

## 7. The headline numbers

In [ ]:
print('--- full dataset run ---')
!sed -n '/=== summary ===/,$p' results/logs/04_full_run.log
print('\n--- gpu versus host reference ---')
!sed -n '/=== gpu versus host reference ===/,$p' results/logs/07_gpu_vs_cpu.log
print('\n--- stream scaling ---')
!grep -E 'streams=|throughput' results/logs/08_stream_scaling.log

## 8. Look at the output

A few inputs beside the edge maps the GPU produced for them.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
from PIL import Image

samples = ['misc_4_1_01', 'misc_5_2_08', 'textures_1_1_01', 'textures_1_5_04']
samples = [s for s in samples if os.path.exists(f'data/output/{s}_edges.png')]
if not samples:
    samples = [os.path.basename(p)[:-10]
               for p in sorted(glob.glob('data/output/*_edges.png'))[:4]]

fig, axes = plt.subplots(2, len(samples), figsize=(4 * len(samples), 8))
for column, stem in enumerate(samples):
    source = glob.glob(f'data/input/{stem}.*')[0]
    axes[0][column].imshow(Image.open(source), cmap='gray')
    axes[0][column].set_title(f'{stem} (input)')
    axes[1][column].imshow(Image.open(f'data/output/{stem}_edges.png'),
                           cmap='gray')
    axes[1][column].set_title('edges')
for axis in axes.ravel():
    axis.axis('off')
plt.tight_layout()
plt.show()

## 9. Download the evidence

`proof_of_execution.tar.gz` is the file to upload to the assignment. The
second download carries the generated images and logs so they can be
committed back to the repository.

In [ ]:
!ls -la results/
!du -sh results/proof_of_execution.tar.gz

# Everything worth committing, in one archive.
!tar -czf /content/artifacts_for_repo.tar.gz results data/output
!du -sh /content/artifacts_for_repo.tar.gz

from google.colab import files

files.download('results/proof_of_execution.tar.gz')
files.download('/content/artifacts_for_repo.tar.gz')